In [18]:
import numpy as np
import pymysql
import pandas as pd
# ==================== 1. 범용 경로 설정 ====================
import sys
import os
from typing import Union
from pathlib import Path

from universal_ts_forecast_function import (
    ensure_datetime_index_df,
    forecast_sarima,
    forecast_ets,
    forecast_theta,
    infer_freq_alias,
    seasonal_periods_from_freq
)

def setup_universal_paths():
    """
    어떤 PC에서도 작동하는 범용 경로 설정
    DATA 폴더를 자동으로 찾아 경로 추가
    """
    current = Path.cwd()

    # 상위 폴더를 탐색하며 DATA 폴더 찾기
    for parent in [current, *current.parents]:
        data_folder = parent / "DATA"
        if data_folder.exists():
            # 프로젝트 루트와 DATA 폴더 모두 추가
            if str(parent) not in sys.path:
                sys.path.insert(0, str(parent))
            if str(data_folder) not in sys.path:
                sys.path.insert(0, str(data_folder))

            print("=" * 70)
            print("📁 경로 설정 완료")
            print("=" * 70)
            print(f"✓ 프로젝트 루트: {parent}")
            print(f"✓ DATA 폴더:    {data_folder}")
            print(f"✓ 현재 위치:     {current}")
            print(f"✓ 운영체제:      {os.name}")
            print("=" * 70 + "\n")

            return {
                'project_root': parent,
                'data_folder': data_folder,
                'current': current
            }

    # 못 찾으면 에러
    raise FileNotFoundError(
        f"❌ DATA 폴더를 찾을 수 없습니다.\n"
        f"현재 위치: {current}\n"
        f"상위 폴더에 DATA 폴더가 있는지 확인하세요."
    )

# 경로 설정 실행
try:
    paths = setup_universal_paths()
except FileNotFoundError as e:
    print(e)
    print("\n대안: 수동으로 경로를 설정하세요.")
    # sys.path.insert(0, "여기에_프로젝트_루트_경로_입력")
    sys.exit(1)

📁 경로 설정 완료
✓ 프로젝트 루트: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
✓ DATA 폴더:    C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA
✓ 현재 위치:     C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\Korea_Market\analysis\한국기업_매출예측
✓ 운영체제:      nt



In [111]:
# ==================== 2. 필요한 모듈 import ====================
try:
    # 예측 함수 import (파일명 확인 필요!)
    from universal_ts_forecast_function import (
        forecast_one_from_pivot_inline,
        monitor_memory_usage
    )
    from stock_invest_function import fetch_table_data

    print("✓ 예측 모듈 import 성공")

except ImportError as e:
    print(f"❌ 모듈 import 실패: {e}")
    print("\n확인 사항:")
    print("1. DATA 폴더에 'universal_ts_forecast_function.py' 파일이 있는가?")
    print("2. DATA 폴더에 'stock_invest_function.py' 파일이 있는가?")
    print("\n파일명이 다르다면 위 import 문을 수정하세요.")
    sys.exit(1)

from DATA.stock_invest_function import *

def get_connection(db_info: dict):
    conn = pymysql.connect(
        host=db_info["host"],
        port=int(db_info["port"]),
        user=db_info["user"],
        password=db_info["password"],
        db=db_info.get("db", db_info.get("database")),
        charset="utf8mb4",
        cursorclass=pymysql.cursors.DictCursor,
    )
    return conn

def extract_revenue_from_fs_df(fs_df: pd.DataFrame,
                               ticker: str,
                               keyword: str = "매출") -> pd.DataFrame:
    """
    korea_fs_data 전체 df(fs_df)에서 특정 ticker의 매출 관련 시계열만 뽑아오는 함수.

    Parameters
    ----------
    fs_df : pd.DataFrame
        fetch_table_data(db_info, "korea_fs_data") 로 가져온 전체 테이블
    ticker : str
        '005930' 또는 'A005930' 둘 다 허용
    keyword : str
        indicator 에 포함될 키워드 (기본값: '매출')

    Returns
    -------
    pd.DataFrame
        date, revenue, indicator 컬럼을 가진 시계열
    """

    # 1) 심볼 통일: 앞에 A 붙이기
    if ticker.startswith("A"):
        symbol = ticker
    else:
        symbol = "A" + ticker

    # 2) 해당 ticker + 매출 관련 indicator 필터링
    mask_symbol = fs_df["symbol"] == symbol
    mask_ind = fs_df["indicator"].astype(str).str.contains(keyword, na=False)

    sub = fs_df.loc[mask_symbol & mask_ind, ["date", "value", "indicator"]].copy()

    if sub.empty:
        print(f"⚠ {symbol} 에 대해 '{keyword}' 를 포함하는 indicator 가 없습니다.")
        print("   우선 아래 코드를 한 번 실행해서 indicator 목록을 눈으로 확인해 보세요:")
        print("   fs_df[fs_df['symbol']=='A005930']['indicator'].unique()")
        return sub

    # 3) 타입 정리
    sub["date"] = pd.to_datetime(sub["date"], errors="coerce")
    sub = sub.dropna(subset=["date"])

    sub["revenue"] = pd.to_numeric(sub["value"], errors="coerce")

    # 출력 형식 정리
    return sub[["date", "revenue", "indicator"]].sort_values("date").reset_index(drop=True)

def fetch_dart_fs_by_ticker_from_db(
    db_info: dict,
    ticker: Union[str, int],
    table_name: str = "korea_fs_data_from_DART",
    verbose: bool = True,
) -> pd.DataFrame:
    database_name = db_info.get("db") or db_info.get("database")

    if isinstance(ticker, int):
        ticker_str = f"{ticker:06d}"
    else:
        ticker_str = str(ticker).zfill(6)

    # 이 부분만 수정 - DictCursor 제거!
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        db=database_name,
        charset="utf8mb4",
    )

    try:
        sql = f"""
            SELECT *
            FROM {table_name}
            WHERE ticker = %s
            ORDER BY bsns_year, reprt_code, quarter, account_id
        """
        df = pd.read_sql(sql, conn, params=[ticker_str])

        if verbose:
            print(f"조회 완료: {len(df)}행, {len(df.columns)}개 컬럼")

        return df
    finally:
        conn.close()

def fetch_revenue_by_ticker(
    db_info: dict,
    ticker: Union[str, int],
    table_name: str = "korea_fs_data_from_DART",
    verbose: bool = True,
) -> pd.DataFrame:
    """
    특정 ticker의 매출(Revenue) 데이터를 추출하는 함수

    Parameters:
    -----------
    db_info : dict
        DB 연결 정보 (host, port, user, password, database)
    ticker : str or int
        종목 코드 (예: '005930' 또는 5930)
    table_name : str
        테이블 이름 (기본값: 'korea_fs_data_from_DART')
    verbose : bool
        진행 상황 출력 여부

    Returns:
    --------
    pd.DataFrame
        매출 데이터 (report_date 순으로 정렬)
        컬럼: quarter, account_id, sj_div, sj_nm, account_nm,
              thstrm_nm, thstrm_amount, report_date, ticker, bsns_year, reprt_code
    """
    database_name = db_info.get("db") or db_info.get("database")
    if database_name is None:
        raise KeyError("db_info 안에 'db' 또는 'database' 키가 없습니다!")

    # ticker 정규화
    if isinstance(ticker, int):
        ticker_str = f"{ticker:06d}"
    else:
        ticker_str = str(ticker).zfill(6)

    if verbose:
        print(f"[INFO] 조회 ticker: {ticker_str}")
        print(f"[INFO] 조회 account_id: ['ifrs_Revenue', 'ifrs-full_Revenue']")

    # DB 연결 (DictCursor 제거)
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        db=database_name,
        charset="utf8mb4",
    )

    try:
        # Revenue 데이터만 조회
        sql = f"""
            SELECT
                quarter,
                account_id,
                sj_div,
                sj_nm,
                account_nm,
                thstrm_nm,
                thstrm_amount,
                report_date,
                ticker,
                bsns_year,
                reprt_code
            FROM {table_name}
            WHERE ticker = %s
                AND account_id IN ('ifrs_Revenue', 'ifrs-full_Revenue')
            ORDER BY report_date, reprt_code, quarter
        """

        df = pd.read_sql(sql, conn, params=[ticker_str])

        if verbose:
            print(f"[INFO] 조회 완료: {len(df)}행")
            if not df.empty:
                print(f"[INFO] 기간: {df['report_date'].min()} ~ {df['report_date'].max()}")
                print(f"[INFO] account_id 분포:")
                print(df['account_id'].value_counts())

        return df

    finally:
        conn.close()

def adjust_fy_to_q4(df: pd.DataFrame) -> pd.DataFrame:
    """
    FY(연간 누적) 데이터를 순수 Q4로 변환

    Logic:
    ------
    각 연도별로:
    - Q1, Q2, Q3: 원본 그대로 (순수 분기 실적)
    - FY: FY - (Q1 + Q2 + Q3) = 순수 Q4

    Parameters:
    -----------
    df : pd.DataFrame
        매출 데이터 (quarter, bsns_year, thstrm_amount 컬럼 필수)

    Returns:
    --------
    pd.DataFrame
        'quarter' 컬럼이 'Q4'로 변경되고
        'thstrm_amount'가 순수 분기 실적으로 조정된 DataFrame
    """
    result_df = df.copy()

    # 연도별 처리
    for year in result_df['bsns_year'].unique():
        year_mask = result_df['bsns_year'] == year
        year_data = result_df[year_mask]

        # FY 행 찾기
        fy_mask = year_mask & (result_df['quarter'] == 'FY')

        if fy_mask.any():
            # FY 금액
            fy_amount = result_df.loc[fy_mask, 'thstrm_amount'].iloc[0]

            # Q1, Q2, Q3 금액 합계
            q123_mask = year_mask & result_df['quarter'].isin(['Q1', 'H1', 'Q3'])
            q123_sum = result_df.loc[q123_mask, 'thstrm_amount'].sum()

            # 순수 Q4 계산
            pure_q4 = fy_amount - q123_sum

            # FY를 Q4로 변경하고 금액 조정
            result_df.loc[fy_mask, 'quarter'] = 'Q4'
            result_df.loc[fy_mask, 'thstrm_amount'] = pure_q4

            print(f"{year}년: FY({fy_amount:,}) - Q1+Q2+Q3({q123_sum:,}) = Q4({pure_q4:,})")

    return result_df


def get_quarterly_revenue_simple(
    db_info: dict,
    ticker: Union[str, int],
    adjust_q4: bool = True
) -> pd.DataFrame:
    """
    매출 데이터 조회 + Q4 조정을 한번에 수행

    Parameters:
    -----------
    db_info : dict
        DB 연결 정보
    ticker : str or int
        종목 코드
    adjust_q4 : bool
        FY를 순수 Q4로 변환할지 여부 (기본값: True)

    Returns:
    --------
    pd.DataFrame
        순수 분기별 매출 데이터
    """
    import pymysql

    # DB 연결 정보
    database_name = db_info.get("db") or db_info.get("database")

    # ticker 정규화
    if isinstance(ticker, int):
        ticker_str = f"{ticker:06d}"
    else:
        ticker_str = str(ticker).zfill(6)

    # DB 연결
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        db=database_name,
        charset="utf8mb4",
    )

    try:
        # 매출 데이터 조회
        sql = """
            SELECT *
            FROM korea_fs_data_from_DART
            WHERE ticker = %s
                AND account_id IN ('ifrs_Revenue', 'ifrs-full_Revenue')
            ORDER BY bsns_year, report_date
        """

        df = pd.read_sql(sql, conn, params=[ticker_str])

        if df.empty:
            print(f"⚠️ {ticker_str} 데이터 없음")
            return df

        print(f"✅ {ticker_str} 매출 데이터 {len(df)}행 조회")

        # Q4 조정
        if adjust_q4:
            df = adjust_fy_to_q4(df)

        return df

    finally:
        conn.close()


def get_all_tickers(db_info: dict) -> list:
    """
    가장 간단한 버전: unique ticker 리스트만 반환

    Parameters:
    -----------
    db_info : dict
        DB 연결 정보

    Returns:
    --------
    list
        unique ticker 리스트 (정렬됨)
    """
    database_name = db_info.get("db") or db_info.get("database")

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        db=database_name,
        charset="utf8mb4",
    )

    try:
        sql = """
            SELECT DISTINCT ticker
            FROM korea_fs_data_from_DART
            ORDER BY ticker
        """

        df = pd.read_sql(sql, conn)
        tickers = df['ticker'].tolist()

        print(f"✅ unique ticker {len(tickers)}개 조회 완료")

        return tickers

    finally:
        conn.close()


✓ 예측 모듈 import 성공


In [73]:
# 예시 db_info 채워 넣으신 뒤 테스트
db_info = {
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'host': get_db_host(),  # 노트북에서는 다른 IP일 수 있음
    'port': 3307,
    'database': 'investar'
}

conn = get_connection(db_info)

# 이미 있으신 코드
fs_df = fetch_table_data(db_info, "korea_fs_data")

✅ 'korea_fs_data' 테이블에서 5902708건의 데이터를 가져왔습니다.


In [112]:
all_tickers = get_all_tickers(db_info)

✅ unique ticker 2451개 조회 완료


['000020',
 '000040',
 '000050',
 '000070',
 '000080',
 '000100',
 '000120',
 '000140',
 '000150',
 '000180']

In [107]:
H = 9   # horizon = 8 steps

# ticker = "058470"
all_tickers = get_all_tickers(db_info)

ticker_dg = 'A' + ticker
revenue_dg = fs_df[(fs_df['symbol'] == ticker_dg) & (fs_df['indicator'] == '매출액(천원)')]
revenue_from_dg = revenue_dg[['date', 'value']].copy()
revenue_from_dg['value'] = revenue_from_dg['value'] * 1000

# 삼성전자 순수 분기별 매출
print("=" * 80)
print(f"{ticker} 순수 분기별 매출")
print("=" * 80)

revenue_df = get_quarterly_revenue_simple(db_info, ticker= ticker)

if not revenue_df.empty:
    # 최근 12개 분기
    recent = revenue_df.tail(12)

    print("\n최근 12개 분기:")
    print(recent[['bsns_year', 'quarter', 'report_date', 'thstrm_amount']].to_string(index=False))

    # 억원 단위로 변환해서 보기
    print("\n최근 8개 분기 (억원):")
    summary = revenue_df.tail(8)[['bsns_year', 'quarter', 'report_date', 'thstrm_amount']].copy()
    summary['매출_억원'] = (summary['thstrm_amount'] / 100_000_000).round(0).astype(int)
    print(summary[['bsns_year', 'quarter', 'report_date', '매출_억원']].to_string(index=False))

revenue_from_dart = revenue_df[['report_date', 'thstrm_amount']].copy()
revenue_from_dg.columns = ['date', 'revenue']
revenue_from_dart.columns = ['date', 'revenue']
revenue_concated_df = pd.concat([revenue_from_dg, revenue_from_dart], axis=0).drop_duplicates(subset=['date'], keep='first').set_index('date')

 # ---- 1) 데이터 준비 -------------------------------------------------------

df = revenue_concated_df.copy()          # 사용자가 제공한 데이터프레임
df = ensure_datetime_index_df(df)        # index = DatetimeIndex
series = df["revenue"].astype(float)

# 최소 데이터 길이 검사
n = len(series)
if n < 48:
    raise ValueError(f"❌ 데이터가 {n}개입니다. 최소 48개 이상 필요합니다.")
else:
    print(f"✅ 데이터 개수 OK: {n}개")


# ---- 2) 빈도(Frequency) 추론 + 계절성 파라미터 -----------------------------

freq = infer_freq_alias(series.index)    # "Q" 혹은 "M" 등
m = seasonal_periods_from_freq(freq)     # 분기=4, 월간=12 자동 설정

print(f"Detected frequency: {freq}  → seasonal_period m = {m}")


# ---- 3) 모델별 예측 (8분기 = 2년) -----------------------------------------

print("\n=== SARIMA 예측 중 ===")
sarima_result = forecast_sarima(
    y=series,
    forecast_horizon=H,
    seasonal_period=m,
    try_transforms=True
)

print("\n=== ETS 예측 중 ===")
ets_result = forecast_ets(
    y=series,
    forecast_horizon=H,
    m=m,
    try_transforms=True
)

print("\n=== Theta 예측 중 ===")
theta_result = forecast_theta(
    y=series,
    forecast_horizon=H,
    m=m,
    try_transforms=True
)


# ---- 4) 결과 합치기 -------------------------------------------------------

forecast_index = pd.date_range(
    start=series.index[-1],
    periods=H+1,
    freq=freq
)[1:]  # 첫 번째는 기존 날짜이므로 제외

result_df = pd.DataFrame({
    "SARIMA": sarima_result.get("forecast"),
    "ETS": ets_result.get("forecast"),
    "Theta": theta_result.get("forecast"),
}, index=forecast_index)

print("\n=== 최종 예측 결과 ===")
print(result_df)

# 앙상블(세 모델 평균) 컬럼 추가
result_df["Ensemble"] = result_df[["SARIMA", "ETS", "Theta"]].mean(axis=1)

result_df['ticker'] = ticker


058470 순수 분기별 매출
✅ 058470 매출 데이터 40행 조회
2015년: FY(99,475,354,810.0) - Q1+Q2+Q3(0.0) = Q4(99,475,354,810.0)
2016년: FY(112,788,733,643.0) - Q1+Q2+Q3(89,532,879,848.0) = Q4(23,255,853,795.0)
2017년: FY(141,510,912,868.0) - Q1+Q2+Q3(110,363,740,855.0) = Q4(31,147,172,013.0)
2018년: FY(150,354,373,050.0) - Q1+Q2+Q3(118,859,416,951.0) = Q4(31,494,956,099.0)
2019년: FY(170,307,404,665.0) - Q1+Q2+Q3(119,157,586,730.0) = Q4(51,149,817,935.0)
2020년: FY(201,335,459,350.0) - Q1+Q2+Q3(159,692,740,338.0) = Q4(41,642,719,012.0)
2021년: FY(280,166,833,067.0) - Q1+Q2+Q3(224,082,148,072.0) = Q4(56,084,684,995.0)
2022년: FY(322,422,803,089.0) - Q1+Q2+Q3(270,774,838,463.0) = Q4(51,647,964,626.0)
2023년: FY(255,573,034,656.0) - Q1+Q2+Q3(197,620,441,583.0) = Q4(57,952,593,073.0)
2024년: FY(278,186,189,427.0) - Q1+Q2+Q3(194,767,344,539.0) = Q4(83,418,844,888.0)

최근 12개 분기:
 bsns_year quarter report_date  thstrm_amount
      2022      Q4  2022-12-31   5.164796e+10
      2023      Q1  2023-03-31   4.909108e+10
      

## DataGuide DATA DB  업로드 (한번만 하면 됨)

In [59]:
# import math
# import pandas as pd
# from sqlalchemy import create_engine, text
#
# def create_fs_long_table_from_df(
#     fs_df: pd.DataFrame,
#     db_info: dict,
#     table_name: str = "korea_fs_long_v2",
#     chunksize: int = 20_000
# ):
#     """
#     fs_df(예: korea_fs_data 전체)를
#     date, ticker, company_name, indicator, value 구조로 변환 후
#     long-format 테이블로 저장.
#     """
#     # 1) 기본 정리
#     print("🔎 원본 fs_df info")
#     print(fs_df.info())
#     print(fs_df.head())
#
#     df = fs_df.copy()
#
#     # symbol -> ticker (A005930 → 005930)
#     df["ticker"] = df["symbol"].astype(str).str.replace("^A", "", regex=True)
#
#     # 타입 정리
#     df["date"] = pd.to_datetime(df["date"], errors="coerce")
#     df["value"] = pd.to_numeric(df["value"], errors="coerce")
#
#     # date 없는 행 제거
#     before_drop = len(df)
#     df = df.dropna(subset=["date"])
#     print(f"✅ date not null: {before_drop:,} → {len(df):,}")
#
#     # 최종 컬럼 순서
#     df = df[["date", "ticker", "company_name", "indicator", "value"]]
#
#     # (중요) 동일 (ticker, indicator, date) 중복 제거 – 마지막 값 기준으로
#     df = df.sort_values(["ticker", "indicator", "date"])
#     before_dup = len(df)
#     df = df.drop_duplicates(subset=["ticker", "indicator", "date"], keep="last")
#     print(f"✅ drop_duplicates: {before_dup:,} → {len(df):,}")
#
#     print("🔎 정리된 df 샘플")
#     print(df.head())
#
#     if df.empty:
#         print("❌ df 가 비어 있습니다. 업로드를 건너뜁니다.")
#         return
#
#     # 2) DB 엔진
#     engine = create_engine(
#         f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
#         f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
#     )
#
#     # 3) 테이블 스키마 생성 + TRUNCATE
#     create_sql = f"""
#     CREATE TABLE IF NOT EXISTS {table_name} (
#         `date`         DATE            NOT NULL,
#         `ticker`       VARCHAR(10)     NOT NULL,
#         `company_name` VARCHAR(100)    NULL,
#         `indicator`    VARCHAR(200)    NOT NULL,
#         `value`        DECIMAL(20,4)   NULL,
#         `created_at`   DATETIME        NOT NULL DEFAULT CURRENT_TIMESTAMP,
#         `updated_at`   DATETIME        NOT NULL DEFAULT CURRENT_TIMESTAMP
#                                       ON UPDATE CURRENT_TIMESTAMP,
#         PRIMARY KEY (ticker, indicator, `date`),
#         KEY idx_indicator_date (indicator, `date`),
#         KEY idx_date (date)
#     );
#     """
#     with engine.begin() as conn:
#         conn.execute(text(create_sql))
#         conn.execute(text(f"TRUNCATE TABLE {table_name};"))
#     print(f"✅ {table_name} 스키마 생성 및 TRUNCATE 완료")
#
#     # 4) chunk 단위 업로드
#     n = len(df)
#     n_chunks = math.ceil(n / chunksize)
#     print(f"➡ 업로드 시작: 총 {n:,}행, chunksize={chunksize}, chunks={n_chunks}")
#
#     for i in range(n_chunks):
#         start = i * chunksize
#         end = min((i + 1) * chunksize, n)
#         chunk = df.iloc[start:end]
#
#         print(f"   - chunk {i+1}/{n_chunks}: {start:,} ~ {end-1:,}  ({len(chunk):,} 행)")
#
#         # 안정성 위해 method=None 으로 (multi가 불안하면)
#         chunk.to_sql(
#             name=table_name,
#             con=engine,
#             if_exists="append",
#             index=False,
#             method=None,
#         )
#
#     print("✅ 전체 업로드 완료")
#
#     # 5) DB에 실제로 몇 행 들어갔는지 확인
#     with engine.begin() as conn:
#         result = conn.execute(text(f"SELECT COUNT(*) FROM {table_name};"))
#         count = list(result)[0][0]
#     print(f"📊 최종 DB row count: {count:,}")


In [60]:
# create_fs_long_table_from_df(fs_df, db_info, table_name="korea_fs_data_from_DG", chunksize=20_000)


🔎 원본 fs_df info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5902708 entries, 0 to 5902707
Data columns (total 5 columns):
 #   Column        Dtype  
---  ------        -----  
 0   symbol        object 
 1   company_name  object 
 2   date          object 
 3   indicator     object 
 4   value         float64
dtypes: float64(1), object(4)
memory usage: 225.2+ MB
None
    symbol company_name        date      indicator         value
0  A000010         조흥은행  2004-01-31  수정PSR(연율화)(배)  4.594400e-01
1  A000010         조흥은행  2004-02-29  수정PSR(연율화)(배)  5.032000e-01
2  A000010         조흥은행  2004-03-31     계속사업이익(천원)  3.612000e+07
3  A000010         조흥은행  2004-03-31      당기순이익(천원)  3.612000e+07
4  A000010         조흥은행  2004-03-31        매출액(천원)  7.920890e+08
✅ date not null: 5,902,708 → 5,902,708
✅ drop_duplicates: 5,902,708 → 5,902,708
🔎 정리된 df 샘플
          date  ticker company_name   indicator     value
90  2005-03-31  000010         조흥은행  YoY_계속사업이익  2.485825
117 2005-06-30  000010    

## FS Data From DART 저장 DB가 너무 이상해서 데이터를 다시 V2 에 다시 업로드

In [70]:
def drop_dart_fs_table_v2(db_info, table_name="korea_fs_data_from_DART_V2"):
    db_name = db_info.get("db") or db_info.get("database")

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        db=db_name,
        charset="utf8mb4",
        cursorclass=pymysql.cursors.DictCursor,
    )

    try:
        with conn.cursor() as cur:
            cur.execute(f"DROP TABLE IF EXISTS {table_name}")
        conn.commit()
        print(f"[INFO] 기존 테이블 {table_name} 삭제 완료.")
    finally:
        conn.close()


def create_dart_fs_table_v2(db_info, table_name="korea_fs_data_from_DART_V2"):
    """
    DART 재무데이터용 새 테이블 생성.
    (ticker, report_date, account_id) 조합이 UNIQUE.
    """
    db_name = db_info.get("db") or db_info.get("database")

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        db=db_name,
        charset="utf8mb4",
        cursorclass=pymysql.cursors.DictCursor,
    )

    create_sql = f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
        id BIGINT AUTO_INCREMENT PRIMARY KEY,

        corp_code      VARCHAR(20)    NOT NULL,
        bsns_year      INT            NOT NULL,
        reprt_code     VARCHAR(20)    NOT NULL,
        quarter        VARCHAR(10)    NOT NULL,

        account_id     VARCHAR(255)   NOT NULL,
        sj_div         VARCHAR(20)    NOT NULL,
        sj_nm          VARCHAR(100)   NOT NULL,
        account_nm     VARCHAR(255)   NOT NULL,
        thstrm_nm      VARCHAR(50)    NOT NULL,

        thstrm_amount  DOUBLE         NULL,
        report_date    DATE           NOT NULL,
        ticker         VARCHAR(20)    NOT NULL,

        UNIQUE KEY uniq_ticker_date_acc (ticker, report_date, account_id),
        KEY idx_ticker_date (ticker, report_date),
        KEY idx_corp_code_acc (corp_code, account_id, report_date)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
    """

    try:
        with conn.cursor() as cur:
            cur.execute(create_sql)
        conn.commit()
        print(f"[INFO] 새 테이블 '{table_name}' 생성 완료.")
    finally:
        conn.close()


import pandas as pd

def normalize_dart_fs_df(dart_fs_df: pd.DataFrame) -> pd.DataFrame:
    """
    V2 스키마에 맞게 기본 타입/결측치 정리.
    """
    df = dart_fs_df.copy()

    str_cols = [
        "corp_code", "reprt_code", "quarter",
        "account_id", "sj_div", "sj_nm",
        "account_nm", "thstrm_nm", "ticker"
    ]
    for col in str_cols:
        if col in df.columns:
            df[col] = df[col].astype(str)

    if "bsns_year" in df.columns:
        df["bsns_year"] = pd.to_numeric(df["bsns_year"], errors="coerce").fillna(0).astype(int)

    if "thstrm_amount" in df.columns:
        df["thstrm_amount"] = pd.to_numeric(df["thstrm_amount"], errors="coerce")

    if "report_date" in df.columns:
        df["report_date"] = pd.to_datetime(df["report_date"], errors="coerce").dt.date

    df = df.dropna(subset=["report_date"])

    return df


import pymysql
import pandas as pd

def insert_dart_fs_df_in_batches(
    dart_fs_df: pd.DataFrame,
    db_info: dict,
    table_name: str = "korea_fs_data_from_DART_V2",
    batch_size: int = 20_000,
):
    """
    정리된 dart_fs_df 를 새 테이블에 batch 단위로 UPSERT.

    (ticker, report_date, account_id)가 같으면
    새로 추가하지 않고 기존 행을 덮어씀.
    """

    db_name = db_info.get("db") or db_info.get("database")

    # DF 내부에서도 일단 중복 제거 (마지막 행 남김)
    key_cols = ["ticker", "report_date", "account_id"]
    if all(c in dart_fs_df.columns for c in key_cols):
        dart_fs_df = (
            dart_fs_df
            .sort_values(key_cols)
            .drop_duplicates(subset=key_cols, keep="last")
        )

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        db=db_name,
        charset="utf8mb4",
        cursorclass=pymysql.cursors.DictCursor,
    )

    cols = [
        "corp_code",
        "bsns_year",
        "reprt_code",
        "quarter",
        "account_id",
        "sj_div",
        "sj_nm",
        "account_nm",
        "thstrm_nm",
        "thstrm_amount",
        "report_date",
        "ticker",
    ]

    insert_sql = f"""
        INSERT INTO {table_name} (
            corp_code,
            bsns_year,
            reprt_code,
            quarter,
            account_id,
            sj_div,
            sj_nm,
            account_nm,
            thstrm_nm,
            thstrm_amount,
            report_date,
            ticker
        ) VALUES (
            %s, %s, %s, %s,
            %s, %s, %s, %s,
            %s, %s, %s, %s
        )
        ON DUPLICATE KEY UPDATE
            corp_code     = VALUES(corp_code),
            bsns_year     = VALUES(bsns_year),
            reprt_code    = VALUES(reprt_code),
            quarter       = VALUES(quarter),
            sj_div        = VALUES(sj_div),
            sj_nm         = VALUES(sj_nm),
            account_nm    = VALUES(account_nm),
            thstrm_nm     = VALUES(thstrm_nm),
            thstrm_amount = VALUES(thstrm_amount);
    """

    try:
        with conn.cursor() as cur:
            total = len(dart_fs_df)
            for start in range(0, total, batch_size):
                end = min(start + batch_size, total)
                chunk = dart_fs_df.iloc[start:end]

                values = []
                for _, row in chunk.iterrows():
                    vals = [row.get(c) for c in cols]
                    vals = [None if pd.isna(v) else v for v in vals]
                    values.append(tuple(vals))

                cur.executemany(insert_sql, values)
                conn.commit()
                print(f"[INFO] {table_name}: {end}/{total} 행까지 UPSERT 완료")

        print(f"[INFO] {table_name} UPSERT 전체 완료.")
    finally:
        conn.close()

import pandas as pd
import pymysql
import numpy as np


def get_revenue_pivot_from_dart_v2(
    db_info: dict,
    ticker: str,
    table_name: str = "korea_fs_data_from_DART_V2",
) -> pd.DataFrame:
    """
    korea_fs_data_from_DART_V2 에서
      - ticker 일치
      - 손익계산서(CIS/IS) 항목 중
      - account_nm에 '매출' 또는 '수익' 이 포함된 행을 매출로 보고

    date(=report_date), revenue(=thstrm_amount/1000) 2컬럼 DataFrame 반환.
    """

    db_name = db_info.get("db") or db_info.get("database")

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        db=db_name,
        charset="utf8mb4",
        cursorclass=pymysql.cursors.DictCursor,
    )

    try:
        sql = f"""
            SELECT
                report_date,
                thstrm_amount,
                account_id,
                account_nm,
                sj_div,
                sj_nm
            FROM {table_name}
            WHERE ticker = %s
              AND sj_div IN ('IS', 'CIS')
              AND (
                     account_nm LIKE '%%매출%%'
                  OR account_nm LIKE '%%수익%%'
                  OR LOWER(account_id) LIKE '%%revenue%%'
              )
              AND thstrm_amount IS NOT NULL
              AND report_date IS NOT NULL
        """
        df = pd.read_sql(sql, conn, params=(ticker,))
    finally:
        conn.close()

    if df.empty:
        print(f"[WARN] ticker={ticker} 에 대해 매출 후보 행을 찾지 못했습니다.")
        return pd.DataFrame(columns=["date", "revenue"])

    # report_date 문자열/DATE 모두 허용
    df["report_date"] = df["report_date"].astype(str)

    # 금액 숫자화 (혹시 모를 문자열 대비)
    df["thstrm_amount"] = (
        df["thstrm_amount"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace(" ", "", regex=False)
        .replace("", np.nan)
    )
    df["thstrm_amount"] = pd.to_numeric(df["thstrm_amount"], errors="coerce")
    df = df.dropna(subset=["thstrm_amount"])

    if df.empty:
        print(f"[WARN] ticker={ticker} 에 대해 숫자로 변환 가능한 thstrm_amount 가 없습니다.")
        return pd.DataFrame(columns=["date", "revenue"])

    # 같은 날짜에 여러 매출계정 있으면 합산
    grouped = (
        df.groupby("report_date", as_index=False)["thstrm_amount"]
          .sum()
    )

    grouped["revenue"] = grouped["thstrm_amount"] / 1000.0

    result = (
        grouped[["report_date", "revenue"]]
        .rename(columns={"report_date": "date"})
        .sort_values("date")
        .reset_index(drop=True)
    )

    return result


In [62]:
# drop_dart_fs_table_v2(db_info, "korea_fs_data_from_DART_V2")

[INFO] 기존 테이블 korea_fs_data_from_DART_V2 삭제 완료.


In [63]:
# create_dart_fs_table_v2(db_info, "korea_fs_data_from_DART_V2")

[INFO] 새 테이블 'korea_fs_data_from_DART_V2' 생성 완료.


In [64]:
# 2) 새 V2 테이블 생성 (UNIQUE 포함)
# create_dart_fs_table_v2(db_info, "korea_fs_data_from_DART_V2")
#
# # 3) DataFrame 타입 정리
# dart_fs_df_norm = normalize_dart_fs_df(dart_fs_df)
#
# # 4) 중복 방지 UPSERT 배치 저장
# insert_dart_fs_df_in_batches(
#     dart_fs_df_norm,
#     db_info,
#     table_name="korea_fs_data_from_DART_V2",
#     batch_size=30_000,
# )

[INFO] 새 테이블 'korea_fs_data_from_DART_V2' 생성 완료.
[INFO] korea_fs_data_from_DART_V2: 30000/7433512 행까지 UPSERT 완료
[INFO] korea_fs_data_from_DART_V2: 60000/7433512 행까지 UPSERT 완료
[INFO] korea_fs_data_from_DART_V2: 90000/7433512 행까지 UPSERT 완료
[INFO] korea_fs_data_from_DART_V2: 120000/7433512 행까지 UPSERT 완료
[INFO] korea_fs_data_from_DART_V2: 150000/7433512 행까지 UPSERT 완료
[INFO] korea_fs_data_from_DART_V2: 180000/7433512 행까지 UPSERT 완료
[INFO] korea_fs_data_from_DART_V2: 210000/7433512 행까지 UPSERT 완료
[INFO] korea_fs_data_from_DART_V2: 240000/7433512 행까지 UPSERT 완료
[INFO] korea_fs_data_from_DART_V2: 270000/7433512 행까지 UPSERT 완료
[INFO] korea_fs_data_from_DART_V2: 300000/7433512 행까지 UPSERT 완료
[INFO] korea_fs_data_from_DART_V2: 330000/7433512 행까지 UPSERT 완료
[INFO] korea_fs_data_from_DART_V2: 360000/7433512 행까지 UPSERT 완료
[INFO] korea_fs_data_from_DART_V2: 390000/7433512 행까지 UPSERT 완료
[INFO] korea_fs_data_from_DART_V2: 420000/7433512 행까지 UPSERT 완료
[INFO] korea_fs_data_from_DART_V2: 450000/7433512 행까지 UPSE

In [71]:
revenue_df = get_revenue_pivot_from_dart_v2(db_info, ticker="000660")
revenue_df

[WARN] ticker=000660 에 대해 숫자로 변환 가능한 thstrm_amount 가 없습니다.


,date,revenue


In [55]:
import pymysql
import pandas as pd


def debug_revenue_rows_v2(
    db_info,
    ticker,
    table_name="korea_fs_data_from_DART_V2",
    limit=50,
):
    """
    korea_fs_data_from_DART_V2 에서
    - 해당 ticker 의 전체 행 개수
    - revenue 추정 계정(account_id에 'revenue') 행 개수
    - 그 중 일부 행(report_date, thstrm_amount, account_id, account_nm) 샘플
    - dtype 및 고유 account_id, thstrm_amount 예시

    를 출력해주는 디버그용 함수.
    """

    db_name = db_info.get("db") or db_info.get("database")

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        db=db_name,
        charset="utf8mb4",
        cursorclass=pymysql.cursors.DictCursor,
    )

    try:
        # 1) 해당 ticker 전체 행 수
        sql_cnt_all = f"""
            SELECT COUNT(*) AS cnt
            FROM {table_name}
            WHERE ticker = %s
        """
        cnt_all = pd.read_sql(sql_cnt_all, conn, params=(ticker,))
        print(f"[INFO] ticker={ticker} 전체 행 수:")
        print(cnt_all)

        # 2) account_id 에 'revenue' 가 포함된 행 수
        sql_cnt_rev = f"""
            SELECT COUNT(*) AS cnt
            FROM {table_name}
            WHERE ticker = %s
              AND LOWER(account_id) LIKE %s
        """
        cnt_rev = pd.read_sql(sql_cnt_rev, conn, params=(ticker, "%revenue%"))
        print(f"\n[INFO] ticker={ticker}, account_id LIKE '%revenue%' 행 수:")
        print(cnt_rev)

        # 3) revenue 후보 행 샘플 가져오기
        sql_sample_rev = f"""
            SELECT
                report_date,
                thstrm_amount,
                account_id,
                account_nm
            FROM {table_name}
            WHERE ticker = %s
              AND LOWER(account_id) LIKE %s
            LIMIT %s
        """
        df_rev = pd.read_sql(sql_sample_rev, conn, params=(ticker, "%revenue%", limit))

        print(f"\n[INFO] ticker={ticker}, revenue 후보 샘플 상위 {limit}행:")
        print(df_rev)

        print("\n[INFO] dtype:")
        print(df_rev.dtypes)

        # 4) account_id / account_nm 고유값 예시
        if not df_rev.empty:
            print("\n[INFO] 고유 account_id / account_nm 예시:")
            print(
                df_rev[["account_id", "account_nm"]]
                .drop_duplicates()
                .head(20)
            )

            # thstrm_amount 원본 값 예시
            print("\n[INFO] thstrm_amount 원본 값 예시:")
            print(df_rev["thstrm_amount"].head(20))

        else:
            print("\n[INFO] revenue 후보(df_rev)가 비었습니다. account_id 전체 예시를 확인합니다.")
            sql_all_sample = f"""
                SELECT
                    report_date,
                    thstrm_amount,
                    account_id,
                    account_nm
                FROM {table_name}
                WHERE ticker = %s
                LIMIT %s
            """
            df_all = pd.read_sql(sql_all_sample, conn, params=(ticker, limit))
            print(df_all)
            print("\n[INFO] dtype (전체 샘플):")
            print(df_all.dtypes)
            print("\n[INFO] 고유 account_id / account_nm 예시:")
            print(
                df_all[["account_id", "account_nm"]]
                .drop_duplicates()
                .head(20)
            )
            print("\n[INFO] thstrm_amount 원본 값 예시:")
            print(df_all["thstrm_amount"].head(20))

    finally:
        conn.close()


In [66]:
debug_revenue_rows_v2(
    db_info,
    ticker="000660",   # 또는 "005930" 등
    table_name="korea_fs_data_from_DART_V2",
    limit=50
)

[INFO] ticker=000660 전체 행 수:
   cnt
0  cnt

[INFO] ticker=000660, account_id LIKE '%revenue%' 행 수:
   cnt
0  cnt

[INFO] ticker=000660, revenue 후보 샘플 상위 50행:
    report_date  thstrm_amount  account_id  account_nm
0   report_date  thstrm_amount  account_id  account_nm
1   report_date  thstrm_amount  account_id  account_nm
2   report_date  thstrm_amount  account_id  account_nm
3   report_date  thstrm_amount  account_id  account_nm
4   report_date  thstrm_amount  account_id  account_nm
5   report_date  thstrm_amount  account_id  account_nm
6   report_date  thstrm_amount  account_id  account_nm
7   report_date  thstrm_amount  account_id  account_nm
8   report_date  thstrm_amount  account_id  account_nm
9   report_date  thstrm_amount  account_id  account_nm
10  report_date  thstrm_amount  account_id  account_nm
11  report_date  thstrm_amount  account_id  account_nm
12  report_date  thstrm_amount  account_id  account_nm
13  report_date  thstrm_amount  account_id  account_nm
14  report_date  

In [65]:
import pymysql

def check_db_connection(db_info, table_name="korea_fs_data_from_DART_V2"):
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        db=db_info.get("db") or db_info.get("database"),
        charset="utf8mb4",
        cursorclass=pymysql.cursors.DictCursor,
    )
    try:
        with conn.cursor() as cur:
            cur.execute("SELECT @@hostname AS host, @@port AS port, DATABASE() AS db")
            info = cur.fetchone()
            print("[INFO] Python이 접속한 DB 정보:")
            print(info)

            cur.execute("SHOW TABLES LIKE %s", (table_name,))
            print("\n[INFO] 이 DB에서 사용하는 테이블 목록(일치 여부 확인):")
            print(cur.fetchall())

            cur.execute(
                f"SELECT report_date, thstrm_amount, account_id, account_nm "
                f"FROM {table_name} LIMIT 5"
            )
            rows = cur.fetchall()
            print("\n[INFO] Python이 보는 V2 테이블 상위 5행:")
            for r in rows:
                print(r)
    finally:
        conn.close()

check_db_connection(db_info, "korea_fs_data_from_DART_V2")


[INFO] Python이 접속한 DB 정보:
{'host': 'HY_Storage', 'port': 3307, 'db': 'investar'}

[INFO] 이 DB에서 사용하는 테이블 목록(일치 여부 확인):
[{'Tables_in_investar (korea_fs_data_from_DART_V2)': 'korea_fs_data_from_DART_V2'}]

[INFO] Python이 보는 V2 테이블 상위 5행:
{'report_date': datetime.date(2018, 12, 31), 'thstrm_amount': -2143162114.0, 'account_id': '-표준계정코드 미사용-', 'account_nm': '확정급여채무의 재측정요소'}
{'report_date': datetime.date(2018, 12, 31), 'thstrm_amount': 0.0, 'account_id': 'dart_AcquisitionOfTreasuryShares', 'account_nm': '자기주식의 취득'}
{'report_date': datetime.date(2018, 12, 31), 'thstrm_amount': 1015521235.0, 'account_id': 'dart_AdjustmentsForAmortisationExpense', 'account_nm': '무형자산상각비'}
{'report_date': datetime.date(2018, 12, 31), 'thstrm_amount': -23575980802.0, 'account_id': 'dart_AdjustmentsForAssetsLiabilitiesOfOperatingActivities', 'account_nm': '영업활동으로인한자산ㆍ부채의변동'}
{'report_date': datetime.date(2018, 12, 31), 'thstrm_amount': 78597531.0, 'account_id': 'dart_AdjustmentsForBadDebtExpenses', 'account_nm':

In [72]:
def fetch_fs_data_by_ticker(db_info: dict,
                            ticker: str,
                            table_name: str = "korea_fs_data_from_DART") -> pd.DataFrame:
    """
    특정 ticker의 재무제표 데이터를 DB에서 조회.
    ticker가 존재하지 않을 경우 메시지 출력 후 빈 DataFrame 반환.
    """

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4"
    )

    try:
        # 먼저 ticker 존재 여부 확인
        check_sql = f"SELECT COUNT(*) AS cnt FROM {table_name} WHERE ticker = %s"
        with conn.cursor() as cur:
            cur.execute(check_sql, (ticker,))
            result = cur.fetchone()
            cnt = result[0]

        if cnt == 0:
            print(f"[INFO] ticker '{ticker}' 는(은) 데이터베이스에 존재하지 않습니다.")
            return pd.DataFrame()   # 빈 DF 반환

        # ticker 존재 → 실제 데이터 조회
        query = f"""
            SELECT
                corp_code,
                bsns_year,
                reprt_code,
                quarter,
                account_id,
                sj_div,
                sj_nm,
                account_nm,
                thstrm_nm,
                thstrm_amount,
                report_date,
                ticker
            FROM {table_name}
            WHERE ticker = %s
            ORDER BY
                bsns_year,
                reprt_code,
                sj_div,
                account_nm
        """

        df = pd.read_sql(query, conn, params=[ticker])
        return df

    finally:
        conn.close()

In [73]:
ticker = "005930"
df = fetch_fs_data_by_ticker(db_info, ticker)

In [74]:
df

,corp_code,bsns_year,reprt_code,quarter,account_id,sj_div,sj_nm,account_nm,thstrm_nm,thstrm_amount,report_date,ticker
0,00126380,2015,11011,FY,ifrs_InvestmentAccountedForUsingEquityMethod,BS,재무상태표,관계기업 및 공동기업 투자,제 47 기,5.276348e+12,2015-12-31,005930
1,00126380,2015,11011,FY,dart_ElementsOfOtherStockholdersEquity,BS,재무상태표,기타자본항목,제 47 기,-1.758045e+13,2015-12-31,005930
2,00126380,2015,11011,FY,dart_ShortTermBorrowings,BS,재무상태표,단기차입금,제 47 기,1.115542e+13,2015-12-31,005930
3,00126380,2015,11011,FY,ifrs_LiabilitiesIncludedInDisposalGroupsClassi...,BS,재무상태표,매각예정분류부채,제 47 기,NaN,2015-12-31,005930
4,00126380,2015,11011,FY,ifrs_NoncurrentAssetsOrDisposalGroupsClassifie...,BS,재무상태표,매각예정분류자산,제 47 기,7.707300e+10,2015-12-31,005930
...,...,...,...,...,...,...,...,...,...,...,...,...
3931,00126380,2025,11014,Q3,ifrs-full_PurchaseOfTreasuryShares,SCE,자본변동표,자기주식의 취득,제 57 기 3분기,0.000000e+00,2025-09-30,005930
3932,00126380,2025,11014,Q3,ifrs-full_Equity,SCE,자본변동표,자본총계,제 57 기 3분기,4.403893e+12,2025-09-30,005930
3933,00126380,2025,11014,Q3,ifrs-full_IncreaseDecreaseThroughSharebasedPay...,SCE,자본변동표,주식기준보상,제 57 기 3분기,0.000000e+00,2025-09-30,005930
3934,00126380,2025,11014,Q3,ifrs-full_GainsLossesOnExchangeDifferencesOnTr...,SCE,자본변동표,해외사업장환산외환차이,제 57 기 3분기,0.000000e+00,2025-09-30,005930
